In [3]:
import os
import json
import yaml
import socket
import traceback
import smtplib
import logging
import pandas as pd
import soccerdata as sd
import undetected_chromedriver as uc
import soccerdata._common as common

from datetime import datetime, timezone
from email.message import EmailMessage
from dataclasses import dataclass
from typing import Optional, Dict, Any, Set
from pydantic import BaseModel
from pymongo import MongoClient, UpdateOne

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

[06/16/26 23:12:46] INFO     No custom team name replacements found. You can configure these in       ]8;id=13914757;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=13914758;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/mario_omescu/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=13914764;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=13914765;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/mario_omescu/soccerdata/config/league_dict.json.                               

In [4]:
CHROME_MAJOR = 147  # must match your installed Chrome major version

def patch_soccerdata_chromedriver() -> None:
    """Patch SoccerData Selenium reader to use undetected_chromedriver."""

    def patched_init_webdriver(self):
        opts = uc.ChromeOptions()
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--start-maximized")
        return uc.Chrome(options=opts, version_main=CHROME_MAJOR)

    common.BaseSeleniumReader._init_webdriver = patched_init_webdriver


def patch_soccerdata_json_loader() -> None:
    """Handle WhoScored responses where JSON is wrapped inside minimal HTML."""

    def tolerant_json_load(fp, *args, **kwargs):
        content = fp.read()

        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="ignore")

        content = content.strip()

        if content.startswith("<html"):
            start = content.find("{")
            end = content.rfind("}") + 1
            if start != -1 and end > start:
                content = content[start:end]

        return json.loads(content)

    json.load = tolerant_json_load


patch_soccerdata_chromedriver()
patch_soccerdata_json_loader()

logger.info("SoccerData patches applied")

                    INFO     SoccerData patches applied                                            ]8;id=13914772;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_4833/2208212149.py\2208212149.py]8;;\:]8;id=13914773;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_4833/2208212149.py#41\41]8;;\

In [5]:
class MongoConfig(BaseModel):
    db: str
    url: str
    collections: Dict[str, str]

class SeasonConfig(BaseModel):
    year: str
    league: str
    name: str
    country: str

class EmailConfig(BaseModel):
    smtp_host: str
    smtp_port: int
    username: str
    from_email: str
    to_email: str
    use_tls: bool = True
    password: Optional[str] = None

class SheduleGamesConfig(BaseModel):
    finished_status_code: int
    required_columns: list[str]

class ScrapeDataConfig(BaseModel):
    mongo: MongoConfig
    season: SeasonConfig
    email: EmailConfig
    schedule_games: SheduleGamesConfig

In [6]:
def load_config(path: str, *, smtp_password_env: str = "SMTP_PASSWORD", smtp_email_env: str = "SMTP_EMAIL") -> ScrapeDataConfig:
    with open(path, "r") as f:
        raw = yaml.safe_load(f)

    email_cfg = EmailConfig(
        **raw["email"],
        password=os.getenv(smtp_password_env),
        username=os.getenv(smtp_email_env),
        from_email=os.getenv(smtp_email_env),
        to_email=os.getenv(smtp_email_env),
    )

    return ScrapeDataConfig(
        mongo=MongoConfig(**raw["mongo"]),
        season=SeasonConfig(**raw["season"]),
        email=email_cfg,
        schedule_games=SheduleGamesConfig(**raw["schedule_games"]),
    )


events_config = load_config("../config/config.yaml")
events_config

ScrapeDataConfig(mongo=MongoConfig(db='WhoScored', url='mongodb://localhost:27017/', collections={'collection_teams': 'available_teams', 'collection_schedule': 'game_schedule', 'collection_logs': 'error_logs', 'collection_raw_events': 'game_raw_events', 'collection_processed_events': 'game_processed_events', 'collection_team_game_stats': 'game_team_stats', 'collection_player_game_stats': 'game_player_stats', 'collection_shot_sequences': 'game_shot_sequences', 'collection_pass_sequences': 'game_pass_sequences'}), season=SeasonConfig(year='2025-2026', league='GER-Bundesliga', name='Bundesliga', country='Germany'), email=EmailConfig(smtp_host='smtp.gmail.com', smtp_port=587, username='omescu.mario.lucian@gmail.com', from_email='omescu.mario.lucian@gmail.com', to_email='omescu.mario.lucian@gmail.com', use_tls=True, password='xubgfeaqjqndfhgr'), schedule_games=SheduleGamesConfig(finished_status_code=6, required_columns=['game_id', 'status', 'start_time', 'home_team_id', 'home_team', 'away_t

In [7]:
client = MongoClient(events_config.mongo.url, serverSelectionTimeoutMS=3000)
client.admin.command("ping")
logger.info("MongoDB connected")

[06/16/26 23:12:48] INFO     MongoDB connected                                                      ]8;id=13914780;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_4833/2772379555.py\2772379555.py]8;;\:]8;id=13914781;file:///var/folders/jq/njqvkrp55b3g2fzq0k92__q80000gn/T/ipykernel_4833/2772379555.py#3\3]8;;\

In [8]:
class ReportError:
    def __init__(self, config: ScrapeDataConfig):
        self.config = config

        self.client = MongoClient(config.mongo.url)
        self.db = self.client[config.mongo.db]

        logs_collection_name = config.mongo.collections["collection_logs"]
        self.collection = self.db[logs_collection_name]

    def report(self, job_name: str, exc: Exception, context: Optional[Dict[str, Any]] = None) -> None:
        context = context or {}

        error_doc = {
            "job": job_name,
            "timestamp": datetime.now(timezone.utc),
            "host": socket.gethostname(),
            "season_year": self.config.season.year,
            "league": self.config.season.league,
            "competition": self.config.season.name,
            "country": self.config.season.country,
            "context": context,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "traceback": traceback.format_exc(),
        }

        try:
            self.collection.insert_one(error_doc)
            logger.info("Error saved to MongoDB")
        except Exception as mongo_error:
            logger.error("Failed to save error to MongoDB: %s", mongo_error, exc_info=True)

        if self.config.email.password:
            self._send_email(error_doc)
        else:
            logger.info("Email not sent: SMTP_PASSWORD is missing")

    def _send_email(self, error_doc: dict) -> None:
        cfg = self.config.email

        subject = f"[WhoScored Scrape ERROR] {error_doc['job']} | {self.config.season.name} {self.config.season.year}"
        body = (
            f"Job: {error_doc['job']}\n"
            f"Time (UTC): {error_doc['timestamp']}\n"
            f"Host: {error_doc['host']}\n\n"
            f"League/Season: {self.config.season.league} / {self.config.season.year}\n"
            f"Competition: {self.config.season.name} ({self.config.season.country})\n\n"
            f"Context: {error_doc['context']}\n\n"
            f"{error_doc['error_type']}: {error_doc['error_message']}\n\n"
            f"Traceback:\n{error_doc['traceback']}\n"
        )

        msg = EmailMessage()
        msg["Subject"] = subject
        msg["From"] = cfg.from_email
        msg["To"] = cfg.to_email
        msg.set_content(body)

        try:
            if cfg.use_tls:
                with smtplib.SMTP(cfg.smtp_host, cfg.smtp_port, timeout=20) as server:
                    server.starttls()
                    server.login(cfg.username, cfg.password)
                    server.send_message(msg)
            else:
                with smtplib.SMTP_SSL(cfg.smtp_host, cfg.smtp_port, timeout=20) as server:
                    server.login(cfg.username, cfg.password)
                    server.send_message(msg)

            logger.info("Error email sent")

        except Exception as email_error:
            logger.exception("Failed to send error email:", repr(email_error))


In [15]:
class GameSchedule:

    def __init__(self, config: ScrapeDataConfig, reporter: ReportError):
        self.config = config
        self.reporter = reporter

        self.ws = sd.WhoScored(
            leagues=config.season.league,
            seasons=config.season.year,
        )

        self.fbref = sd.FBref(
            leagues=config.season.league,
            seasons=config.season.year,
        )

        self.client = MongoClient(config.mongo.url)
        self.db = self.client[config.mongo.db]

        schedule_collection_name = config.mongo.collections["collection_schedule"]
        self.collection_schedule = self.db[schedule_collection_name]
        teams_collection_name = config.mongo.collections["collection_teams"]
        self.collection_teams = self.db[teams_collection_name]

    def _read_team_mapping(self) -> dict[int, str]:
        cursor = self.collection_teams.find(
            {},
            {"_id": 0, "ws_team_id": 1, "fbref_team_name": 1},
        )

        return {
            int(doc["ws_team_id"]): doc["fbref_team_name"]
            for doc in cursor
        }

    def _read_season_schedule(self) -> pd.DataFrame:
        return self.ws.read_schedule(force_cache=False).reset_index(drop=False)

    def _read_fbref_schedule(self) -> pd.DataFrame:

        df = self.fbref.read_schedule().reset_index(drop=False)
        # return df
        df["fbref_date"] = pd.to_datetime(df["date"]).dt.date

        return df[["week", "fbref_date", "home_team", "away_team", "referee", "attendance", "venue"]].rename(
            columns={
                "home_team": "fbref_home_team",
                "away_team": "fbref_away_team",
                "week": "week",
            }
        )

    def _extract_finished_game_ids(self) -> Set[int]:
        cursor = self.collection_schedule.find(
            {"game_status": "finished"},
            {"_id": 0, "game_id": 1},
        )
        return {int(doc["game_id"]) for doc in cursor if doc.get("game_id") is not None}

    def _get_newly_finished_games(self) -> pd.DataFrame:
        
        df = self._read_season_schedule()
        
        team_map = self._read_team_mapping()

        df["match_date"] = pd.to_datetime(df["start_time"]).dt.date
        df["fbref_home_team"] = df["home_team_id"].astype(int).map(team_map)
        df["fbref_away_team"] = df["away_team_id"].astype(int).map(team_map)

        df_fbref = self._read_fbref_schedule()

        df = df.merge(
            df_fbref,
            left_on=["match_date", "fbref_home_team", "fbref_away_team"],
            right_on=["fbref_date", "fbref_home_team", "fbref_away_team"],
            how="left",
        )

        required = set(events_config.schedule_games.required_columns)
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"WhoScored schedule missing columns: {sorted(missing)}")

        finished_ids = self._extract_finished_game_ids()

        df_remaining = df[~df["game_id"].astype(int).isin(finished_ids)].copy()
        df_new_finished = df_remaining[
            df_remaining["status"] == events_config.schedule_games.finished_status_code].copy()

        if df_new_finished.empty:
            return pd.DataFrame()

        return df_new_finished.reset_index(drop=True)

    def save_schedule(self) -> None:
        job_name = "GameSchedule.save_schedule"

        try:
            df_new_finished = self._get_newly_finished_games()
            if df_new_finished.empty:
                logger.info("No new finished games found.")
                return

            ops = []

            for _, row in df_new_finished.iterrows():
                game_id = int(row["game_id"])
                start_time = row.get("start_time")
                week = int(row.get("week"))

                if pd.notna(start_time):
                    start_time = pd.to_datetime(start_time).to_pydatetime()

                doc = {
                    "game_id": game_id,
                    "game_date": start_time,
                    "season": self.config.season.year,
                    "week": week,
                    "competition_name": self.config.season.name,
                    "competition_country": self.config.season.country,
                    "home_team_id": row.get("home_team_id"),
                    "home_team_name": row.get("home_team"),
                    "away_team_id": row.get("away_team_id"),
                    "away_team_name": row.get("away_team"),
                    "game_status": "finished",
                    'referee': row.get('referee'),
                    'attendance': row.get('attendance'),
                    'venue': row.get('venue')
                }

                ops.append(
                    UpdateOne(
                        {"game_id": game_id},
                        {"$set": doc},
                        upsert=True,
                    )
        )
            result = self.collection_schedule.bulk_write(ops, ordered=False)
            affected = result.upserted_count + result.modified_count

            logger.info("Finished games processed: %s", len(df_new_finished))
            logger.info("Mongo affected documents: %s", affected)
            logger.info("Saved/updated %s documents", affected)
            client.close()

        except Exception as e:
            self.reporter.report(
                job_name=job_name,
                exc=e,
                context={
                    "season_year": self.config.season.year,
                    "league": self.config.season.league,
                },
            )
            raise

In [16]:
reporter = ReportError(config=events_config)

game_schedule = GameSchedule(
    config=events_config,
    reporter=reporter,
)

# game_schedule.save_schedule()

[06/16/26 23:14:27] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/WhoScored     ]8;id=13914838;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13914839;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

                    INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=13914844;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=13914845;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

[06/16/26 23:14:29] INFO     Saving cached data to /Users/mario_omescu/soccerdata/data/FBref         ]8;id=13914850;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=13914851;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

[06/16/26 23:14:30] INFO     patching driver executable /Users/mario_omescu/Library/Application      ]8;id=13914856;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py\patcher.py]8;;\:]8;id=13914857;file:///opt/homebrew/Caskroom/miniconda/base/envs/sports_analytics/lib/python3.11/site-packages/undetected_chromedriver/patcher.py#346\346]8;;\
                             Support/undetected_chromedriver/undetected_chromedriver                               

In [17]:
game_schedule._read_fbref_schedule()

,week,fbref_date,fbref_home_team,fbref_away_team,referee,attendance,venue
0,1,2025-08-22,Bayern Munich,RB Leipzig,Florian Badstübner,75000,Allianz Arena
1,1,2025-08-23,Eintracht Frankfurt,Werder Bremen,Harm Osmers,59500,Deutsche Bank Park
2,1,2025-08-23,Freiburg,Augsburg,Felix Zwayer,33600,Europa-Park Stadion
3,1,2025-08-23,Heidenheim,Wolfsburg,Benjamin Brand,13000,Voith-Arena
4,1,2025-08-23,Leverkusen,Hoffenheim,Daniel Siebert,29390,BayArena
...,...,...,...,...,...,...,...
303,34,2026-05-16,St Pauli,Wolfsburg,Daniel Siebert,29546,Millerntor-Stadion
304,34,2026-05-16,Union Berlin,Augsburg,Patrick Ittrich,22012,Stadion An der Alten Försterei
305,34,2026-05-16,Werder Bremen,Dortmund,Bastian Dankert,42000,Weserstadion
306,<NA>,2026-05-21,Wolfsburg,Paderborn 07,<NA>,<NA>,Volkswagen Arena
